# generator-project-and-reshape — worked example 3: Project to a 4x4 seed then upsample to 16x16

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `generator-project-and-reshape`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

After the projection-and-reshape seed, a DCGAN stacks `ConvTranspose2d(kernel_size=4, stride=2, padding=1)` blocks, each of which exactly doubles the spatial size. Starting from a 4x4 seed, two such upsamples reach 16x16. Tracking each intermediate shape is the whole verification.

## Worked solution

We build a tiny generator module and follow the shapes.

1. `self.project = nn.Linear(latent_dim, base_C * 4 * 4)` produces the flat seed features.
2. In `forward`, `view(B, base_C, 4, 4)` turns them into a 4x4 spatial seed.
3. `up1 = ConvTranspose2d(base_C, base_C//2, kernel_size=4, stride=2, padding=1)` doubles 4x4 to 8x8. The output size formula `(in-1)*stride - 2*padding + kernel = (4-1)*2 - 2 + 4 = 8` confirms the doubling.
4. `up2 = ConvTranspose2d(base_C//2, out_C, 4, 2, 1)` doubles 8x8 to 16x16, landing at the requested output channels.
5. We return all three intermediate tensors so the test can assert each shape: seed `(B, base_C, 4, 4)`, mid `(B, base_C//2, 8, 8)`, out `(B, out_C, 16, 16)`.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(2)

class Generator416(nn.Module):
    def __init__(self, latent_dim, base_C=64, out_C=3):
        super().__init__()
        self.base_C = base_C
        self.project = nn.Linear(latent_dim, base_C * 4 * 4)
        self.up1 = nn.ConvTranspose2d(base_C, base_C // 2, kernel_size=4, stride=2, padding=1)
        self.up2 = nn.ConvTranspose2d(base_C // 2, out_C, kernel_size=4, stride=2, padding=1)

    def forward(self, z):
        B = z.shape[0]
        seed = self.project(z).view(B, self.base_C, 4, 4)
        mid = self.up1(seed)
        out = self.up2(mid)
        return {'seed': seed, 'mid': mid, 'out': out}

g = Generator416(latent_dim=100, base_C=64, out_C=3)
res = g(t.randn(2, 100))
print({k: tuple(v.shape) for k, v in res.items()})